# Import Libraries & Dependencies

In [ ]:
!pip install -q nltk rouge-score pycocoevalcap

In [ ]:
import os, json, torch
from tqdm.notebook import tqdm
import torch
import transformers
from transformers import GPT2LMHeadModel
import datasets
import accelerate
import nltk

from lora import inject_lora
from generate import generate_batch
import evaluate

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

# Table 11
BEAM_SIZE = 10      
LENGTH_PENALTY = 0.9      
NO_REPEAT_NGRAM = 4       
MAX_NEW_TOKENS = 128
GEN_BATCH_SIZE = 8

## Load Datasets

In [ ]:
# CODE TO LOAD DATASET TOKENIZED WITH GPT2-MEDIUM
tokModel = "gpt2-medium"

In [ ]:
# CODE TO LOAD DATASET TOKENIZED WITH GPT2-MEDIUM
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import GPT2Tokenizer

def load_e2e(tokModel):
    base = "https://raw.githubusercontent.com/tuetschek/e2e-dataset/master/"
    tokenizer = GPT2Tokenizer.from_pretrained(tokModel)
    tokenizer.pad_token = tokenizer.eos_token

    def rename(df):
        return df.rename(columns={"mr": "input", "ref": "label"})

    def tokenize(batch):
        return tokenizer(
            batch["input"],
            text_target=batch["label"],
            truncation=True,
            max_length=512,
        )

    dataset = DatasetDict({
        "train":      Dataset.from_pandas(rename(pd.read_csv(base + "trainset.csv"))),
        "validation": Dataset.from_pandas(rename(pd.read_csv(base + "devset.csv"))),
        "test":       Dataset.from_pandas(rename(pd.read_csv(base + "testset_w_refs.csv"))),
    })

    return dataset.map(tokenize, batched=True)

dataset = load_e2e(tokModel)

In [ ]:
# CODE TO LOAD DATASET TOKENIZED WITH GPT2-LARGE

## Inject LoRA

In [ ]:
# CODE TO INJECT LORA INTO GPT2-MEDIUM

#Load model and Inject LoRA
model = GPT2LMHeadModel.from_pretrained("gpt2-medium")
model = inject_lora(model, rank=4, alpha=32)
model.eval()
print(f"LoRA injected successfully")

# Count how many trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable params: {trainable:,}")

In [ ]:
# # CODE TO INJECT LORA INTO GPT2-LARGE

# #Load model and Inject LoRA
# model = GPT2LMHeadModel.from_pretrained("gpt2-large")
# model = inject_lora(model, rank=4, alpha=32)
# model.eval()
# print(f"LoRA injected successfully")

# # Count how many trainable params
# trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
# print(f"trainable params: {trainable:,}")

## Training

In [ ]:
# CODE TO TRAIN LORA GPT2-MEDIUM
tokenizer = GPT2Tokenizer.from_pretrained(tokModel)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

def preprocess(batch):
    input_ids, labels = [], []
    for prompt, target in zip(batch["input"], batch["label"]):
        prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
        target_ids = tokenizer(target + tokenizer.eos_token, add_special_tokens=False)["input_ids"]
        input_ids.append((prompt_ids + target_ids)[:512])
        labels.append(([-100] * len(prompt_ids) + target_ids)[:512])
    return {"input_ids": input_ids, "labels": labels}

train_dataset = dataset["train"].map(preprocess, batched=True, remove_columns=dataset["train"].column_names)

args = transformers.TrainingArguments(
    output_dir="artifacts/gpt2-medium-lora",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = transformers.Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    data_collator=transformers.DataCollatorForSeq2Seq(tokenizer, model=model, padding=True),
)

trainer.train()
os.makedirs(args.output_dir, exist_ok=True)
torch.save({name: param.detach().cpu() for name, param in model.named_parameters() if param.requires_grad}, os.path.join(args.output_dir, "lora_weights.pt"))
model.eval()

In [ ]:
# CODE TO TRAIN LORA GPT2-LARGE

# Generate Outputs For Testing

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Create Tokenizer (If not already available)
# tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
# tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = 'left'  


# Load Tokenized Test Prompts
with open(os.path.join(TEST_DATA_PATH)) as f:
    prompts = json.load(f)

print(f'Loaded {len(prompts)} test prompts')
print(f'Sample prompt: {prompts[0][:80]}...')

In [ ]:
# Run generation over all test prompts
all_predictions = []
for i in tqdm(range(0, len(prompts), GEN_BATCH_SIZE), desc='Generating'):
    batch = prompts[i : i + GEN_BATCH_SIZE]
    all_predictions.extend(generate.generate_batch(model, tokenizer, batch, device))

print(f'\nGenerated {len(all_predictions)} predictions')
print('\nSample outputs:')
for i in range(min(3, len(all_predictions))):
    print(f'  [{i}] Prompt: {prompts[i][:60]}...')
    print(f'       Output: {all_predictions[i]}')
    print()

In [ ]:
# Save predictions
with open(OUTPUT_PATH, 'w') as f:
    for pred in all_predictions:
        f.write(pred + '\n')

print(f'Saved predictions')

# Evaluating Test Results

Computes all five NLG metrics from Table 3 of the paper:

| Metric  | What it measures |
|---------|-----------------|
| BLEU    | N-gram precision (1–4) with brevity penalty |
| NIST    | Like BLEU but weights rarer n-grams more heavily |
| METEOR  | Unigram F-score with stemming + WordNet synonyms (needs `nltk wordnet`) |
| ROUGE-L | Longest common subsequence F-score |
| CIDEr   | TF-IDF-weighted n-gram cosine similarity |

In [ ]:
# FILL OUT ACCORDINGLY
PREDICTIONS_FILE = '' # PREDICTION OF TEST DATASET GENERATED by train.ipynb USING generate.py
REFERENCES_FILE  = '' # DICTIONARY MAPPING THE MULTIPLE REF PER MR
OUTPUT_FILE      = '' # LOCATION TO SAVE RESULTS

os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

# Load predictions (one per line, produced by train.ipynb using generate.py)
with open(PREDICTIONS_FILE) as f:
    predictions = [line.strip() for line in f if line.strip()]

# Load references (dict: MR -> [ref1, ref2, ...])
with open(REFERENCES_FILE) as f:
    ref_dict = json.load(f)

# Align: references[i] is the list of refs for predictions[i]
references = list(ref_dict.values())

In [ ]:
print('Computing metrics...')
bleu = evaluate.compute_bleu(predictions, references);    print(f'  BLEU:    {bleu}')
nist = evaluate.compute_nist(predictions, references);    print(f'  NIST:    {nist}')
meteor = evaluate.compute_meteor(predictions, references);  print(f'  METEOR:  {meteor}')
rouge_l = evaluate.compute_rouge_l(predictions, references); print(f'  ROUGE-L: {rouge_l}')
cider = evaluate.compute_cider(predictions, references);   print(f'  CIDEr:   {cider}')

In [ ]:
results = {
    'num_examples':    len(predictions),
    'predictions_file': PREDICTIONS_FILE,
    'BLEU':    bleu,
    'NIST':    nist,
    'METEOR':  meteor,
    'ROUGE-L': rouge_l,
    'CIDEr':   cider,
}

with open(OUTPUT_FILE, 'w') as f:
    json.dump(results, f, indent=2)

print(f'Results saved to: {OUTPUT_FILE}')